# 04 — Interventions\n\nTrying progressively stronger fixes. Each step builds on the previous one.\n\n- 4a: photometric augmentation (unsupervised, uses target imagery)\n- 4b: self-training with mean teacher\n- 4c: label 30 actively selected target chips + random control\n- Oracle: all 240 target labels (ceiling)\n- 4c from baseline: skip the ladder, test if 4a/4b matter when you have labels

In [ ]:
import sys, os, copy, json
sys.path.insert(0, "..")
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from pathlib import Path

from src.data_acquisition import CLASS_NAMES, NUM_CLASSES, NODATA_LABEL, compute_dataset_stats
from src.dataset import LandCoverDataset, load_norm_stats
from src.training import create_model, evaluate, get_device, TrainConfig, train_model, compute_class_weights
from src.diagnostics import compute_quantiles, collect_raw_pixels
from src.interventions import (
    PhotoAugDataset, precompute_chip_quantiles,
    run_self_training, SelfTrainConfig,
    select_active_chips, select_random_chips, MixedDomainDataset,
    compute_chip_entropy,
)

torch.manual_seed(42)
np.random.seed(42)

SOURCE_DIR = Path("../data/source")
TARGET_DIR = Path("../data/target")
CHECKPOINT_DIR = Path("../checkpoints")
FIG_DIR = Path("../report/figures")
device = get_device("mps")

with open(SOURCE_DIR / "splits.json") as f:
    src_splits = json.load(f)
with open(TARGET_DIR / "splits.json") as f:
    tgt_splits = json.load(f)

mean, std = load_norm_stats(SOURCE_DIR / "norm_stats.json")
tgt_train = tgt_splits["train"]
tgt_eval = tgt_splits["eval"]

# Critical: verify no overlap between eval and train
assert not set(tgt_eval) & set(tgt_train), "Target eval and train sets overlap!"
print(f"Verified: target eval ({len(tgt_eval)}) and train ({len(tgt_train)}) don't overlap")

# Precompute quantiles for histogram-match augmentation (using train chips only)
src_pixels = collect_raw_pixels(SOURCE_DIR, src_splits["train"], max_pixels=500_000)
src_quantiles = compute_quantiles(src_pixels)
tgt_chip_quantiles = precompute_chip_quantiles(TARGET_DIR, tgt_train)
print(f"Precomputed {len(tgt_chip_quantiles)} target chip quantiles for hist-match augmentation")

# Eval loaders and class weights
src_stats = compute_dataset_stats(SOURCE_DIR, src_splits["train"])
class_weights = compute_class_weights(src_stats["class_freq"])

tgt_eval_ds = LandCoverDataset(TARGET_DIR, tgt_eval, mean, std, augment=False)
tgt_eval_loader = DataLoader(tgt_eval_ds, batch_size=8, shuffle=False, num_workers=8)

src_val_ds = LandCoverDataset(SOURCE_DIR, src_splits["val"], mean, std, augment=False)
src_val_loader = DataLoader(src_val_ds, batch_size=8, shuffle=False, num_workers=8)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device), ignore_index=NODATA_LABEL)

all_results = {}
BASELINE_SRC = 0.561
BASELINE_TGT = 0.063
print(f"Device: {device}")

## 4a: Photometric / target-style augmentation retrain\n\nFine-tune baseline with W-distance-calibrated augmentations + histogram matching to target train chips.\nThis uses target imagery (unsupervised) — not purely source-only.

In [ ]:
model_4a = create_model(encoder_name="resnet34", in_channels=5, num_classes=NUM_CLASSES)
model_4a.load_state_dict(torch.load(CHECKPOINT_DIR / "best_model.pth", map_location=device, weights_only=True))

aug_ds = PhotoAugDataset(
    SOURCE_DIR, src_splits["train"], mean, std,
    target_chip_quantiles=tgt_chip_quantiles, src_quantiles=src_quantiles,
)
aug_loader = DataLoader(aug_ds, batch_size=8, shuffle=True, num_workers=8, drop_last=True)

config_4a = TrainConfig(epochs=15, lr=1e-4, save_dir=str(CHECKPOINT_DIR / "4a"),
                        device="mps", num_workers=8)
history_4a = train_model(model_4a, aug_loader, src_val_loader, config_4a, class_weights=class_weights)

# Evaluate best 4a
model_4a.load_state_dict(torch.load(CHECKPOINT_DIR / "4a" / "best_model.pth", map_location=device, weights_only=True))
model_4a = model_4a.to(device)
_, _, tgt_miou_4a, _, _ = evaluate(model_4a, tgt_eval_loader, criterion, device)
_, _, src_miou_4a, _, _ = evaluate(model_4a, src_val_loader, criterion, device)

print(f"\n4a: src_val={src_miou_4a:.4f}, tgt_eval={tgt_miou_4a:.4f}")
all_results["4a"] = {"src": float(src_miou_4a), "tgt": float(tgt_miou_4a)}

## 4b: Self-training with mean teacher\n\nMean teacher (EMA α=0.99) with per-class confidence thresholds.\nBoth teacher and student see the same geo-augmented target images (no asymmetry on target side).\nSource branch uses photometric augmentation. Pseudo-label accuracy logged but never used for selection.\nSaves last-epoch teacher (fixed schedule, not best-target-eval).

In [ ]:
# Initialize student from 4a
model_4b = create_model(encoder_name="resnet34", in_channels=5, num_classes=NUM_CLASSES)
model_4b.load_state_dict(torch.load(CHECKPOINT_DIR / "4a" / "best_model.pth", map_location=device, weights_only=True))

# Source loader with photometric augs (student sees augmented)
src_aug_ds = PhotoAugDataset(
    SOURCE_DIR, src_splits["train"], mean, std,
    target_chip_quantiles=tgt_chip_quantiles, src_quantiles=src_quantiles,
)
src_aug_loader = DataLoader(src_aug_ds, batch_size=8, shuffle=True, num_workers=8, drop_last=True)

# Target train loader (teacher sees clean, student sees same images)
# Labels loaded from disk but only used for pseudo-label accuracy logging
tgt_train_ds = LandCoverDataset(TARGET_DIR, tgt_train, mean, std, augment=True)
tgt_train_loader = DataLoader(tgt_train_ds, batch_size=8, shuffle=True, num_workers=8, drop_last=True)

st_config = SelfTrainConfig(
    epochs=15, lr=5e-5, ema_alpha=0.99,
    confidence_quantile=0.7, lambda_u_max=0.5, rampup_epochs=3,
    save_dir=str(CHECKPOINT_DIR / "4b"), device="mps", num_workers=8,
)

history_4b, teacher_4b = run_self_training(
    model_4b, src_aug_loader, tgt_train_loader, src_val_loader, tgt_eval_loader,
    st_config, class_weights=class_weights,
)

# Evaluate last-epoch teacher
teacher_4b.eval()
_, _, tgt_miou_4b, _, _ = evaluate(teacher_4b, tgt_eval_loader, criterion, device)
_, _, src_miou_4b, _, _ = evaluate(teacher_4b, src_val_loader, criterion, device)

print(f"\n4b: src_val={src_miou_4b:.4f}, tgt_eval={tgt_miou_4b:.4f}")
print(f"Pseudo-label accuracy (last epoch): {history_4b['pseudo_accuracies'][-1]:.3f}")
print(f"Pseudo-label coverage (last epoch): {history_4b['pseudo_coverages'][-1]:.3f}")
all_results["4b"] = {"src": float(src_miou_4b), "tgt": float(tgt_miou_4b),
                      "pseudo_acc": history_4b["pseudo_accuracies"],
                      "pseudo_cov": history_4b["pseudo_coverages"]}

## 4c: Active selection of 30 labeled target chips + random control (3 seeds)

In [ ]:
# Active selection using 4b teacher
teacher_4b.eval()
active_30 = select_active_chips(
    teacher_4b, TARGET_DIR, tgt_train, mean, std, device, n_select=30, seed=42,
)
print(f"Active 30: {sorted(active_30)[:10]}...")

# --- Active fine-tune from 4b ---
model_4c = create_model(encoder_name="resnet34", in_channels=5, num_classes=NUM_CLASSES)
model_4c.load_state_dict(torch.load(CHECKPOINT_DIR / "4b" / "best_model.pth", map_location=device, weights_only=True))

active_ds = MixedDomainDataset(SOURCE_DIR, src_splits["train"], TARGET_DIR, active_30,
                               mean, std, target_oversample=8, augment=True)
active_loader = DataLoader(active_ds, batch_size=8, shuffle=True, num_workers=8, drop_last=True)

config_4c = TrainConfig(epochs=10, lr=5e-5, save_dir=str(CHECKPOINT_DIR / "4c_active"), device="mps", num_workers=8)
train_model(model_4c, active_loader, src_val_loader, config_4c, class_weights=class_weights)

model_4c.load_state_dict(torch.load(CHECKPOINT_DIR / "4c_active" / "best_model.pth", map_location=device, weights_only=True))
model_4c = model_4c.to(device)
_, _, tgt_miou_4c_a, _, _ = evaluate(model_4c, tgt_eval_loader, criterion, device)
_, _, src_miou_4c_a, _, _ = evaluate(model_4c, src_val_loader, criterion, device)
print(f"\n4c Active: src_val={src_miou_4c_a:.4f}, tgt_eval={tgt_miou_4c_a:.4f}")
all_results["4c_active"] = {"src": float(src_miou_4c_a), "tgt": float(tgt_miou_4c_a)}

# --- Random control: 3 seeds ---
random_tgt_mious = []
random_src_mious = []
for seed in [42, 43, 44]:
    random_30 = select_random_chips(tgt_train, n_select=30, seed=seed)
    model_r = create_model(encoder_name="resnet34", in_channels=5, num_classes=NUM_CLASSES)
    model_r.load_state_dict(torch.load(CHECKPOINT_DIR / "4b" / "best_model.pth", map_location=device, weights_only=True))

    random_ds = MixedDomainDataset(SOURCE_DIR, src_splits["train"], TARGET_DIR, random_30,
                                   mean, std, target_oversample=8, augment=True)
    random_loader = DataLoader(random_ds, batch_size=8, shuffle=True, num_workers=8, drop_last=True)
    config_r = TrainConfig(epochs=10, lr=5e-5, save_dir=str(CHECKPOINT_DIR / f"4c_random_{seed}"), device="mps", num_workers=8)
    train_model(model_r, random_loader, src_val_loader, config_r, class_weights=class_weights)

    model_r.load_state_dict(torch.load(CHECKPOINT_DIR / f"4c_random_{seed}" / "best_model.pth", map_location=device, weights_only=True))
    model_r = model_r.to(device)
    _, _, tgt_m, _, _ = evaluate(model_r, tgt_eval_loader, criterion, device)
    _, _, src_m, _, _ = evaluate(model_r, src_val_loader, criterion, device)
    random_tgt_mious.append(tgt_m)
    random_src_mious.append(src_m)
    print(f"  Random (seed={seed}): src={src_m:.4f}, tgt={tgt_m:.4f}")

random_tgt_mean = np.mean(random_tgt_mious)
random_tgt_std = np.std(random_tgt_mious)
random_src_mean = np.mean(random_src_mious)
print(f"\n4c Random (3 seeds): src={random_src_mean:.4f}, tgt={random_tgt_mean:.4f} ± {random_tgt_std:.4f}")
all_results["4c_random"] = {"src": float(random_src_mean), "tgt": float(random_tgt_mean),
                             "tgt_std": float(random_tgt_std), "seeds": random_tgt_mious}

## Reference rows: Oracle (all 240 target labels) + 4c from baseline (skip ladder)

In [ ]:
# --- Oracle: fine-tune baseline on ALL 240 target labels ---
model_oracle = create_model(encoder_name="resnet34", in_channels=5, num_classes=NUM_CLASSES)
model_oracle.load_state_dict(torch.load(CHECKPOINT_DIR / "best_model.pth", map_location=device, weights_only=True))

oracle_ds = MixedDomainDataset(SOURCE_DIR, src_splits["train"], TARGET_DIR, tgt_train,
                               mean, std, target_oversample=1, augment=True)
oracle_loader = DataLoader(oracle_ds, batch_size=8, shuffle=True, num_workers=8, drop_last=True)

config_oracle = TrainConfig(epochs=15, lr=1e-4, save_dir=str(CHECKPOINT_DIR / "oracle"), device="mps", num_workers=8)
train_model(model_oracle, oracle_loader, src_val_loader, config_oracle, class_weights=class_weights)

model_oracle.load_state_dict(torch.load(CHECKPOINT_DIR / "oracle" / "best_model.pth", map_location=device, weights_only=True))
model_oracle = model_oracle.to(device)
_, _, tgt_miou_oracle, _, _ = evaluate(model_oracle, tgt_eval_loader, criterion, device)
_, _, src_miou_oracle, _, _ = evaluate(model_oracle, src_val_loader, criterion, device)
print(f"Oracle (all 240): src={src_miou_oracle:.4f}, tgt={tgt_miou_oracle:.4f}")
all_results["oracle"] = {"src": float(src_miou_oracle), "tgt": float(tgt_miou_oracle)}

# --- 4c from baseline directly (skip 4a/4b) ---
model_4c_base = create_model(encoder_name="resnet34", in_channels=5, num_classes=NUM_CLASSES)
model_4c_base.load_state_dict(torch.load(CHECKPOINT_DIR / "best_model.pth", map_location=device, weights_only=True))

# Use same active_30 chips but fine-tune from baseline
base_active_ds = MixedDomainDataset(SOURCE_DIR, src_splits["train"], TARGET_DIR, active_30,
                                    mean, std, target_oversample=8, augment=True)
base_active_loader = DataLoader(base_active_ds, batch_size=8, shuffle=True, num_workers=8, drop_last=True)

config_base = TrainConfig(epochs=10, lr=5e-5, save_dir=str(CHECKPOINT_DIR / "4c_from_baseline"), device="mps", num_workers=8)
train_model(model_4c_base, base_active_loader, src_val_loader, config_base, class_weights=class_weights)

model_4c_base.load_state_dict(torch.load(CHECKPOINT_DIR / "4c_from_baseline" / "best_model.pth", map_location=device, weights_only=True))
model_4c_base = model_4c_base.to(device)
_, _, tgt_miou_4c_base, _, _ = evaluate(model_4c_base, tgt_eval_loader, criterion, device)
_, _, src_miou_4c_base, _, _ = evaluate(model_4c_base, src_val_loader, criterion, device)
print(f"4c from baseline: src={src_miou_4c_base:.4f}, tgt={tgt_miou_4c_base:.4f}")
all_results["4c_from_baseline"] = {"src": float(src_miou_4c_base), "tgt": float(tgt_miou_4c_base)}

## Results Summary

In [ ]:
# Full results table
rows = [
    ("Baseline (source-only)", BASELINE_SRC, BASELINE_TGT),
    ("+ per-chip std (probe)", 0.404, 0.107),
    ("+ hist match (probe)", 0.561, 0.102),
]
for name, key in [
    ("4a: aug retrain (unsup.)", "4a"),
    ("4b: + self-training", "4b"),
    ("4c: + 30 active labels", "4c_active"),
    ("4c: 30 active from baseline", "4c_from_baseline"),
    ("Oracle: all 240 labels", "oracle"),
]:
    if key in all_results:
        rows.append((name, all_results[key]["src"], all_results[key]["tgt"]))

if "4c_random" in all_results:
    r = all_results["4c_random"]
    rows.append((f"4c: 30 random (mean±std)", r["src"], r["tgt"]))

print("=" * 75)
print(f"{'Method':<35} {'Source mIoU':>12} {'Target mIoU':>12} {'Delta':>10}")
print("-" * 75)
for name, src, tgt in rows:
    delta = f"+{(tgt - BASELINE_TGT)*100:.1f} pts" if tgt != BASELINE_TGT else "--"
    tgt_str = f"{tgt:.3f}"
    if "random" in name.lower() and "4c_random" in all_results:
        tgt_str = f"{all_results['4c_random']['tgt']:.3f}±{all_results['4c_random']['tgt_std']:.3f}"
    print(f"{name:<35} {src:>12.3f} {tgt_str:>12} {delta:>10}")
print("=" * 75)

# Bar chart
fig, ax = plt.subplots(figsize=(14, 5))
labels_plot = [r[0][:20] for r in rows]
tgt_vals = [r[2] for r in rows]
colors = ["#D94A4A", "#FFB347", "#FFB347", "#66BB6A", "#43A047", "#2E7D32",
          "#1B5E20", "#7B1FA2", "#81C784"][:len(tgt_vals)]

bars = ax.bar(range(len(tgt_vals)), tgt_vals, color=colors, edgecolor="black", linewidth=0.5)
for i, (bar, v) in enumerate(zip(bars, tgt_vals)):
    ax.annotate(f"{v:.3f}", (bar.get_x() + bar.get_width()/2, v),
                ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_xticks(range(len(labels_plot)))
ax.set_xticklabels(labels_plot, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Target mIoU")
ax.set_title("Interventions Ladder: Target mIoU")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(FIG_DIR / "04_results_ladder.png", dpi=150, bbox_inches="tight")
plt.show()

# Save results
with open(CHECKPOINT_DIR / "intervention_results.json", "w") as f:
    json.dump({k: {kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                   for kk, vv in v.items()} for k, v in all_results.items()}, f, indent=2)
print(f"Saved to {CHECKPOINT_DIR / 'intervention_results.json'}")